In [3]:
from google.colab import files
uploaded = files.upload()

KeyboardInterrupt: 

In [4]:
import zipfile

zip_path = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("/content/crypto_data")

data_path = "/content/crypto_data"

In [5]:
import os

data_path = "/content/crypto_data"

files = os.listdir(data_path)
print("All files:\n", files)

All files:
 ['coin_Tether.csv', 'coin_NEM.csv', 'coin_Uniswap.csv', 'coin_Dogecoin.csv', 'coin_Polkadot.csv', 'coin_Bitcoin.csv', 'coin_Iota.csv', 'coin_Tron.csv', 'coin_USDCoin.csv', 'coin_EOS.csv', 'coin_XRP.csv', 'coin_WrappedBitcoin.csv', 'coin_Cosmos.csv', 'coin_Aave.csv', 'coin_Stellar.csv', 'coin_Cardano.csv', 'coin_ChainLink.csv', 'coin_Ethereum.csv', 'coin_Litecoin.csv', 'coin_Solana.csv', 'coin_BinanceCoin.csv', 'coin_Monero.csv', 'coin_CryptocomCoin.csv']


In [6]:
csv_files = [f for f in files if f.endswith('.csv')]
print("CSV files:", csv_files)

CSV files: ['coin_Tether.csv', 'coin_NEM.csv', 'coin_Uniswap.csv', 'coin_Dogecoin.csv', 'coin_Polkadot.csv', 'coin_Bitcoin.csv', 'coin_Iota.csv', 'coin_Tron.csv', 'coin_USDCoin.csv', 'coin_EOS.csv', 'coin_XRP.csv', 'coin_WrappedBitcoin.csv', 'coin_Cosmos.csv', 'coin_Aave.csv', 'coin_Stellar.csv', 'coin_Cardano.csv', 'coin_ChainLink.csv', 'coin_Ethereum.csv', 'coin_Litecoin.csv', 'coin_Solana.csv', 'coin_BinanceCoin.csv', 'coin_Monero.csv', 'coin_CryptocomCoin.csv']


OBJECTIVE 1 — Data Collection & Preprocessing

In [7]:
import pandas as pd

test_file = csv_files[0]
df_test = pd.read_csv(os.path.join(data_path, test_file))

print(test_file)
print(df_test.head())
print(df_test.columns)

coin_Tether.csv
   SNo    Name Symbol                 Date      High       Low      Open  \
0    1  Tether   USDT  2015-02-26 23:59:59  1.212320  1.194710  1.210420   
1    2  Tether   USDT  2015-03-02 23:59:59  0.607890  0.568314  0.571249   
2    3  Tether   USDT  2015-03-03 23:59:59  0.606229  0.604416  0.605129   
3    4  Tether   USDT  2015-03-06 23:59:59  1.000000  1.000000  1.000000   
4    5  Tether   USDT  2015-03-07 23:59:59  1.000000  1.000000  1.000000   

      Close        Volume    Marketcap  
0  1.205740      5.955460  303364.1840  
1  0.606502      3.032500  152595.9032  
2  0.606229      3.031130  152527.2164  
3  1.000000     92.647202  251600.0000  
4  1.000000  58196.800781  251600.0000  
Index(['SNo', 'Name', 'Symbol', 'Date', 'High', 'Low', 'Open', 'Close',
       'Volume', 'Marketcap'],
      dtype='object')


In [8]:
from sklearn.preprocessing import MinMaxScaler

def preprocess_coin(file_path):
    try:
        df = pd.read_csv(file_path)

        df.columns = df.columns.str.strip().str.lower()

        if 'date' not in df.columns or 'close' not in df.columns:
            return None, None

        df.rename(columns={'date': 'Date', 'close': 'Close'}, inplace=True)

        df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
        df = df.dropna(subset=['Date'])

        df = df.sort_values('Date')
        df.set_index('Date', inplace=True)

        df = df.ffill().dropna()

        if len(df) < 50:
            return None, None

        df['Daily_Return'] = df['Close'].pct_change()
        df['MA7'] = df['Close'].rolling(7).mean()
        df['MA30'] = df['Close'].rolling(30).mean()
        df['Volatility'] = df['Daily_Return'].rolling(7).std()

        df.dropna(inplace=True)

        scaler = MinMaxScaler()
        scaled = scaler.fit_transform(df[['Close','MA7','MA30','Volatility']])

        scaled_df = pd.DataFrame(
            scaled,
            columns=['Close','MA7','MA30','Volatility'],
            index=df.index
        )

        return df, scaled_df

    except Exception as e:
        print("Error in file:", file_path, e)
        return None, None

In [9]:
coin_data = {}

for file in csv_files:
    coin_name = file.split('.')[0]
    path = os.path.join(data_path, file)

    df, scaled_df = preprocess_coin(path)

    if df is not None:
        coin_data[coin_name] = {
            "original": df,
            "scaled": scaled_df
        }

print("Total valid coins processed:", len(coin_data))

Total valid coins processed: 23


In [11]:
import os

os.makedirs("/content/processed", exist_ok=True)

for coin, data in coin_data.items():
    df = data["original"]   # ✅ FIX HERE
    df.to_csv(f"/content/processed/{coin}_processed.csv")

print("Saved processed files")

Saved processed files


In [15]:
import shutil

shutil.make_archive('/content/processed_files', 'zip', '/content/processed')

'/content/processed_files.zip'

In [16]:
from google.colab import files

files.download('/content/processed_files.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

OBJECTIVE 2 — Price Prediction Models

In [12]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

results = {}

for coin, data in coin_data.items():
    df = data["scaled"]

    train_size = int(len(df) * 0.8)
    train = df[:train_size]
    test = df[train_size:]

    X_train = train[['MA7','MA30','Volatility']]
    y_train = train['Close']

    X_test = test[['MA7','MA30','Volatility']]
    y_test = test['Close']

    lr = LinearRegression().fit(X_train, y_train)
    rf = RandomForestRegressor(n_estimators=50).fit(X_train, y_train)

    y_lr = lr.predict(X_test)
    y_rf = rf.predict(X_test)

    results[coin] = {
        "LR_RMSE": np.sqrt(mean_squared_error(y_test, y_lr)),
        "RF_RMSE": np.sqrt(mean_squared_error(y_test, y_rf)),
        "LR_MAE": mean_absolute_error(y_test, y_lr),
        "RF_MAE": mean_absolute_error(y_test, y_rf),
    }

In [17]:
import pandas as pd

results_df = pd.DataFrame(results).T
results_df = results_df.sort_values(by="RF_RMSE")

results_df

,LR_RMSE,RF_RMSE,LR_MAE,RF_MAE
coin_USDCoin,0.013103,0.012205,0.009725,0.006640
coin_Tether,0.011923,0.013674,0.006654,0.008740
coin_NEM,0.017574,0.024096,0.009245,0.011259
coin_Iota,0.022812,0.033754,0.013731,0.018400
coin_EOS,0.031496,0.034675,0.017451,0.019932
coin_Monero,0.029693,0.036258,0.016674,0.021738
coin_Stellar,0.031562,0.038981,0.016403,0.022213
coin_XRP,0.020593,0.046439,0.010149,0.017970
coin_Litecoin,0.031776,0.047676,0.017508,0.023175
coin_Tron,0.038517,0.074863,0.024414,0.039831


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14,6))

top_coins = list(coin_data.keys())[:5]   # ✅ FIX

for coin in top_coins:
    df = coin_data[coin]["original"]

    normalized = df['Close'] / df['Close'].iloc[0]
    plt.plot(normalized, label=coin)

plt.legend()
plt.grid(True)
plt.title("Normalized Cryptocurrency Price Trends (Comparative Analysis)")
plt.ylabel("Normalized Price")
plt.xlabel("Date")
plt.show()

In [ ]:
combined = pd.DataFrame()

for coin, data in coin_data.items():
    combined[coin] = data["original"]['Close']

combined = combined.ffill().dropna()

corr = combined.corr()
corr

In [ ]:
top_coins = list(results_df.index[:2])
print("Top coins for LSTM:", top_coins)